# Balanceamento de Amostras

Exercicio da Alura: selecionar 50 sobrenomes de cada uma das 18 nacionalidades.

In [ ]:
import os
import string
import unicodedata
import numpy as np
import torch

arquivos = sorted(
    arq for arq in os.listdir('data/names')
    if arq.endswith('.txt')
)
categorias = [os.path.splitext(arq)[0] for arq in arquivos]

def normalizar(nome):
    nome = unicodedata.normalize('NFKD', nome)
    nome = ''.join(c for c in nome if not unicodedata.combining(c))
    return ''.join(c for c in nome if c in string.ascii_letters + " .'-")

dados = []
rotulos = []
for arquivo, categoria in zip(arquivos, categorias):
    with open(os.path.join('data/names', arquivo), encoding='utf-8-sig') as f:
        nomes = [normalizar(linha.strip()) for linha in f]
    nomes = [nome for nome in nomes if nome]
    dados.append(nomes)
    rotulos.append([categoria] * len(nomes))

print('Categorias:', len(categorias))
print('Total de sobrenomes:', sum(map(len, dados)))


## Conversao para tensores

Cada letra recebe uma representacao One-hot. Cada nacionalidade recebe um indice inteiro.

In [ ]:
dicionario = string.ascii_letters + " .'-"

def name2tensor(nome):
    nome = normalizar(nome)
    tensor = torch.zeros(len(nome), len(dicionario))
    for k, letra in enumerate(nome):
        tensor[k, dicionario.index(letra)] = 1
    return tensor

def label2tensor(rotulo):
    return torch.LongTensor([categorias.index(rotulo)])

print('Exemplo:', dados[0][0])
print('Tensor:', name2tensor(dados[0][0]).shape)
print('Rotulo:', label2tensor(rotulos[0][0]))


## Amostragem balanceada

A cada chamada selecionamos 50 indices por categoria. A selecao e feita com reposicao, preservando as duplicatas.

In [ ]:
def amostrar(num_amostras=50):
    dados_amostra = []
    rotulos_amostra = []

    for indice, categoria in enumerate(categorias):
        num_nomes = len(dados[indice])
        if num_nomes == 0:
            raise ValueError(f'Categoria vazia: {categoria}')

        indices = np.random.choice(
            range(num_nomes), size=num_amostras, replace=True
        )

        for posicao in indices:
            dados_amostra.append(name2tensor(dados[indice][posicao]))
            rotulos_amostra.append(label2tensor(categoria))

    return dados_amostra, rotulos_amostra

dados_tns, rotulos_tns = amostrar(50)
print('Total de amostras:', len(dados_tns))
print('Total de rotulos:', len(rotulos_tns))
print('Primeiro tensor:', dados_tns[0].shape)
print('Primeiro rotulo:', rotulos_tns[0])


## Verificacao do balanceamento

Confirmamos que cada nacionalidade aparece exatamente 50 vezes.

In [ ]:
from collections import Counter

contagem = Counter(int(rotulo.item()) for rotulo in rotulos_tns)
for indice, quantidade in sorted(contagem.items()):
    print(f'{categorias[indice]}: {quantidade}')

assert len(dados_tns) == 50 * len(categorias)
assert len(rotulos_tns) == len(dados_tns)
assert all(quantidade == 50 for quantidade in contagem.values())
print('BALANCEAMENTO VALIDADO')


In [ ]:
# AULA: IMPLEMENTANDO EM PYTORCH - RNNCELL
import torch
from torch import nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

class RNN(nn.Module):
    def __init__(self, tam_entrada, tam_hidden, tam_saida):
        super().__init__()
        self.tam_hidden = tam_hidden
        self.recorrente = nn.RNNCell(tam_entrada, tam_hidden)
        self.saida = nn.Linear(tam_hidden, tam_saida)
        self.ativacao = nn.LogSoftmax(dim=1)

    def forward(self, nome):
        h = torch.zeros(
            1, self.tam_hidden,
            device=nome.device,
            dtype=nome.dtype
        )

        for letra in nome:
            h = self.recorrente(letra.unsqueeze(0), h)

        return self.ativacao(self.saida(h))

modelo = RNN(
    len(dicionario),
    256,
    len(categorias)
).to(device)

nome_teste = dados_tns[0].to(device)

with torch.no_grad():
    resultado = modelo(nome_teste)

print("Dispositivo:", device)
print("Entrada:", nome_teste.shape)
print("Saida:", resultado.shape)

assert resultado.shape == (1, len(categorias))
assert torch.allclose(
    resultado.exp().sum(dim=1),
    torch.ones(1, device=device),
    atol=1e-5
)

print("RNN VALIDADA")
